# 01 · Configuración, población y definición del desenlace — Muerte fetal, MIMIC-IV (M3)

| Campo | Contenido |
|---|---|
| Secuencia del framework | **01: E1 · E2 · E3 · P1** → 02: E4 → 03: E5 · E6 → R3 → 04: R3 · E2 · E3 · P1 · E4 → 05: E5 · E6 · E7 · P2 → 06: P3 · clase · E11 |
| Qué hace | Muestra la configuración (E1), cuenta el universo obstétrico y la población (E2), verifica el catálogo de códigos de muerte fetal y la tabla de etiqueta por admisión (E3), y comprueba que se reproduzcan los conteos de referencia de la auditoría (P1) |
| Etapas y puertas | E1 · E2 · E3 · P1 |
| Decisiones | D1 (base de datos), D3 (población), D5 (definición del desenlace); ver `docs/outcome_card_fetal_death_es.md` |
| Entradas | MIMIC-IV v3.1 en PostgreSQL y la cohorte obstétrica revisada (`cohort_obstetric_v2` y sus tablas acompañantes) |
| Tablas del esquema mimiciv | `fetal_death_code_patterns_v2` (`sql/01_labels/01_fetal_death_codes_official_es.sql`) y `fetal_death_label_v2` (`sql/01_labels/02_fetal_death_label_by_admission_es.sql`) |
| Salidas | `data/outputs_01_outcome_definition.txt` (solo conteos agregados; de 1 a 4 se muestran como "<5") |
| Tiempo | Alrededor de un minuto si las tablas ya existen |
| Anterior / siguiente | — / `02_count_audit` |
| Versión en inglés | `01_outcome_definition.ipynb` (mismo código) |

> **Reconstruir las tablas.** Las tablas del esquema `mimiciv` que usa este notebook se leen como están y solo se corren sus verificaciones. Si una tabla no existe, se construye con su script SQL. Para reconstruirla (por ejemplo, si cambia un script de `sql/`), poner **`REBUILD = True`** en la primera celda de código y volver a correr el notebook.

In [8]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_01_outcome_definition.txt"
engine = get_connect_database()

## E1 · Gobierno y configuración — fuente, acuerdo de uso, semilla, software y regla de publicación

Se fija antes de correr cualquier análisis: la fuente de datos y su versión, el acuerdo de uso, la semilla aleatoria, las versiones del software, los conteos de referencia que deben reproducirse y la regla de publicación. La celda muestra esa configuración y comprueba que la base de datos responde, que existen las tablas de la cohorte obstétrica revisada y que los paquetes están instalados.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| La base responde, existen las tablas de la cohorte y los paquetes están instalados (ok) | Se pasa a E2 | Copia en PostgreSQL de MIMIC-IV v3.1 |
| No hay conexión con la base de datos | Revisar `.env` y que PostgreSQL esté activo (README) | — |
| Falta una tabla de la cohorte | Construir primero la cohorte obstétrica revisada (README, requisitos previos) | — |

In [9]:
# Stage S1: the configuration every notebook of this repository uses, shown so it can be audited.
import platform
def version(module):
    try:
        return importlib.import_module(module).__version__
    except Exception:
        return "NOT INSTALLED"
try:
    with engine.connect() as _con:
        _con.exec_driver_sql("SELECT 1")
    db_status = "ok"
except Exception:
    db_status = "NO CONNECTION: check .env and that PostgreSQL is running"
UPSTREAM = ["cohort_obstetric_v2", "hadm_delivery_v2", "obstetric_dx_v2", "predictors_24h_v2"]
up_missing = [t for t in UPSTREAM if db_status != "ok" or not table_exists(engine, t)]
rows = [("data source", "MIMIC-IV v3.1 (PhysioNet), Beth Israel Deaconess Medical Center, 2008-2022; dates shifted per patient", "fixed"),
        ("database", "local PostgreSQL copy of MIMIC-IV (schema mimiciv); credentials in .env (not versioned)", db_status),
        ("revised obstetric cohort", ", ".join(UPSTREAM), "ok" if not up_missing else "MISSING: " + ", ".join(up_missing)),
        ("data use", "PhysioNet Credentialed Health Data Use Agreement and CITI training: research only; no re-identification; "
                     "data not shared or redistributed", "rule"),
        ("publication rule", "only aggregate outputs (data/outputs_*.txt); counts from 1 to 4 shown as <5 (D13); "
                             "no subject_id or hadm_id published", "rule"),
        ("random seed", "20261119 (notebooks 05 and 06)", "fixed"),
        ("reference counts", "frozen at the count audit (count_audit.md) and reproduced by gate G1 in notebooks 01 and 04", "fixed"),
        ("python", platform.python_version(), "info")]
rows += [(m, version(m), "ok" if version(m) != "NOT INSTALLED" else "INSTALL (requirements.txt)")
         for m in ["pandas", "numpy", "sklearn", "scipy", "sqlalchemy", "psycopg2"]]
show("S1 · Configuration of this repository", pd.DataFrame(rows, columns=["item", "value", "status"]))
assert db_status == "ok" and not up_missing, "S1: fix the configuration before going on"

,ítem,valor,estado
1,fuente de datos,"MIMIC-IV v3.1 (PhysioNet), Beth Israel Deaconess Medical Center, 2008-2022; fechas desplazadas por paciente",fijo
2,base de datos,copia local de MIMIC-IV en PostgreSQL (esquema mimiciv); credenciales en .env (no versionado),ok
3,cohorte obstétrica revisada,"cohort_obstetric_v2, hadm_delivery_v2, obstetric_dx_v2, predictors_24h_v2",ok
4,uso de los datos,Acuerdo de uso de datos con credenciales de PhysioNet y curso CITI: solo investigación; sin reidentificación; datos no compartidos ni redistribuidos,regla
5,regla de publicación,solo salidas agregadas (data/outputs_*.txt); conteos de 1 a 4 como <5 (D13); no se publica ningún subject_id ni hadm_id,regla
6,semilla aleatoria,20261119 (notebooks 05 y 06),fijo
7,conteos de referencia,fijados en la auditoría de conteos (count_audit_es.md) y reproducidos por la puerta P1 en los notebooks 01 y 04,fijo
8,python,3.11.9,información
9,pandas,3.1.1,ok
10,numpy,2.4.6,ok


## E2 · Universo y población — admisiones obstétricas y población del modelo (decisión D3)

El universo es la cohorte obstétrica revisada: admisiones de mujeres de 12 a 55 años con un diagnóstico de embarazo, parto o puerperio. La población del modelo son las hospitalizaciones de parto más las admisiones con un código de mortinato que la marca de parto dejó fuera (decisión D3).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Los conteos coinciden con la auditoría de conteos | La población queda fija: se pasa a E3 | 19.440 admisiones de 13.735 pacientes |
| Un conteo difiere | Revisar la cohorte obstétrica revisada antes de seguir | — |

In [10]:
# Stage S2: obstetric universe and population of the model (decision D3), on the revised obstetric cohort
pop = pd.read_sql("""
    SELECT COUNT(*)                                                               AS n_admissions,
           COUNT(DISTINCT subject_id)                                             AS n_patients,
           COUNT(*) FILTER (WHERE is_delivery_v2)                                 AS n_delivery_hospitalizations,
           COUNT(*) FILTER (WHERE has_stillbirth_code AND NOT is_delivery_v2)     AS n_stillbirth_code_without_delivery_flag,
           COUNT(*) FILTER (WHERE is_delivery_v2 OR has_stillbirth_code)          AS n_population,
           COUNT(DISTINCT subject_id) FILTER (WHERE is_delivery_v2 OR has_stillbirth_code) AS n_patients_population
    FROM   mimiciv.cohort_obstetric_v2""", engine)
show("S2 · Obstetric universe and population of the model", pop)
p0 = pop.iloc[0]
decision("S2 · Decision D3: population", [
    ("population of the model", f"{int(p0.n_population):,} admissions of {int(p0.n_patients_population):,} patients",
     "delivery hospitalizations (v2 flag) or a stillbirth code"),
    ("stillbirth codes without the delivery flag", f"{int(p0.n_stillbirth_code_without_delivery_flag)} admissions",
     "they stay in the population: the delivery flag leaves them out because they also carry an abortive code"),
    ("record and grouping", "one row per admission", "one record per delivery hospitalization, grouped by patient (S5, notebook 03)")])

,admisiones,pacientes,hospitalizaciones de parto,código de mortinato sin marca de parto,población,pacientes de la población
1,"25,736","15,084","19,425",15,"19,440","13,735"


,revisión,resultado,decisión
1,población del modelo,"19,440 admisiones de 13,735 pacientes",hospitalizaciones de parto (marca v2) o un código de mortinato
2,códigos de mortinato sin marca de parto,15 admisiones,se quedan en la población: la marca de parto las deja fuera porque también tienen un código de aborto
3,registro y agrupación,una fila por admisión,"un registro por hospitalización de parto, agrupado por paciente (E5, notebook 03)"


## E3 · Definición del desenlace — catálogo verificado de códigos de muerte fetal (tarea T15)

El desenlace es un sustituto: un código diagnóstico de muerte fetal en la admisión de la madre. MIMIC-IV no tiene un vínculo madre–recién nacido, así que no captura las muertes neonatales tempranas. El catálogo lista cada patrón de código con su título oficial y su nivel: *estricto* (muerte fetal de todo el embarazo), *parcial* (embarazo múltiple en el que murió algún feto) y *excluido* (muerte fetal temprana y códigos del registro del recién nacido, listados solo para auditoría).

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| El catálogo tiene los 18 patrones previstos y ningún código estricto queda fuera del universo | El catálogo queda fijo: con él se construye la tabla de etiqueta | Estrictos 4 + 4, parciales 3 + 2, excluidos 2 + 3 (CIE-10 + CIE-9) |
| Falta un patrón o un código queda fuera del universo | Corregir el catálogo (`sql/01_labels/01_...`) y reconstruir | — |

In [11]:
# Stage S3: verified catalogue of fetal-death codes (task T15): mimiciv.fetal_death_code_patterns_v2
res15 = ensure_table(engine, "fetal_death_code_patterns_v2", SQL / "01_labels" / "01_fetal_death_codes_official.sql",
                     REBUILD, log=say)
show_results(res15, ["Q-T15a patterns by tier", "Q-T15b admissions per pattern", "Q-T15c full codes in the universe",
                     "Q-T15e excluded families in the universe"])

01_fetal_death_codes_official.sql: 5 sentencias corridas (solo verificaciones), 3 omitidas, 0 errores, 21.3 s


,nivel,versión CIE,patrones
1,excluido,9,<5
2,excluido,10,<5
3,parcial,9,<5
4,parcial,10,<5
5,estricto,9,<5
6,estricto,10,<5


,nivel,versión CIE,patrón de código,admisiones en toda la base,en el universo obstétrico,en la población D3
1,excluido,9,632,69,69,<5
2,excluido,10,O021,44,44,6
3,parcial,9,V273,7,7,7
4,parcial,10,O312,12,12,8
5,parcial,10,Z373,5,5,<5
6,estricto,9,6564,44,44,38
7,estricto,9,V271,47,47,47
8,estricto,9,V274,5,5,5
9,estricto,9,V277,<5,<5,<5
10,estricto,10,O364,45,45,44


,nivel,versión CIE,código CIE,admisiones
1,parcial,9,V273,7
2,parcial,10,O3121X1,<5
3,parcial,10,O3122X1,<5
4,parcial,10,O3122X2,<5
5,parcial,10,O3123X0,<5
6,parcial,10,O3123X1,5
7,parcial,10,O3123X2,<5
8,parcial,10,Z373,5
9,estricto,9,65641,38
10,estricto,9,65643,6


,familia,admisiones,también con código de evento,sin código de evento
1,pérdida temprana (O02.1 / 632),113,11,102


## E3 · Definición del desenlace — tabla de etiqueta por admisión (tarea T16) y decisión D5

Una fila por admisión del universo. Además de la etiqueta principal, guarda cada definición de sensibilidad como columna (códigos parciales, 22 y 28 semanas o más, sin códigos de pérdida temprana), la posición del código, las semanas de gestación de los códigos Z3A y la era de la CIE. La salida termina con la tabla «Decisión D5», que muestra el resultado de la revisión y el camino tomado.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Semanas de gestación solo en CIE-10 y pérdidas previables dentro de la etiqueta | Etiqueta principal: código estricto a cualquier edad gestacional; las demás como sensibilidad | D5: 129 eventos estrictos; 142 con códigos parciales; 55 con 22 semanas o más |
| La etiqueta no tiene momento del evento | Se registra como riesgo conocido: el momento de los eventos se mide en E6 (notebook 03) | 65 de 129 eventos tienen el código de muerte fetal como diagnóstico principal |

In [12]:
# Stage S3: label table by admission (task T16): mimiciv.fetal_death_label_v2
res16 = ensure_table(engine, "fetal_death_label_v2", SQL / "01_labels" / "02_fetal_death_label_by_admission.sql",
                     REBUILD, log=say)
show_results(res16, ["Q-T16b strict events outside population D3", "Q-T16c strict events by code family",
                     "Q-T16d position of the strict code", "Q-T16e events with early-loss code", "Q-T16f label sizes",
                     "Q-T16g D8 admissions shorter than 6 h"])
sizes = res16["Q-T16f label sizes"].set_index("scope").iloc[0]
decision("S3 · Decision D5: outcome definition", [
    ("main label", f"{int(sizes.n_main_strict)} strict events", "strict fetal-death code at any gestational age"),
    ("strict plus partial codes", f"{int(sizes.n_strict_plus_partial)} events", "sensitivity analysis"),
    ("strict without an early-loss code", f"{int(sizes.n_strict_without_early_loss_code)} events", "sensitivity analysis"),
    ("22 weeks or more (ICD-10 only)", f"{int(sizes.n_ge22_weeks)} events", "sensitivity analysis"),
    ("28 weeks or more (ICD-10 only)", f"{int(sizes.n_ge28_weeks)} events", "sensitivity analysis"),
    ("strict events without gestational weeks", f"{int(sizes.n_strict_weeks_unknown)} events", "reported apart"),
    ("time of the event", "none: discharge codes without a date", "measured with proxies in S6 (notebook 03)")])

02_fetal_death_label_by_admission.sql: 8 sentencias corridas (solo verificaciones), 4 omitidas, 0 errores, 0.2 s


,era CIE,código de muerte intrauterina,código de desenlace de mortinato,código 656.43 (anteparto),siguiente admisión de parto,eventos
1,solo CIE-10,Sí,No,No,4 sin admisión de parto posterior,<5
2,solo CIE-9,Sí,No,Sí,2 de 8 a 90 días,<5
3,solo CIE-9,Sí,No,Sí,3 más de 90 días,<5
4,solo CIE-9,Sí,No,Sí,4 sin admisión de parto posterior,<5


,era CIE,en la población D3,familia de código,eventos,con 656.43 (anteparto)
1,solo CIE-10,No,1 solo código de muerte intrauterina,<5,0
2,solo CIE-10,Sí,1 solo código de muerte intrauterina,<5,0
3,solo CIE-10,Sí,2 solo código de desenlace de mortinato,27,0
4,solo CIE-10,Sí,3 ambas familias,43,0
5,solo CIE-9,No,1 solo código de muerte intrauterina,6,6
6,solo CIE-9,Sí,1 solo código de muerte intrauterina,<5,0
7,solo CIE-9,Sí,2 solo código de desenlace de mortinato,20,0
8,solo CIE-9,Sí,3 ambas familias,34,0


,familia de código,posición,eventos
1,1 solo código de muerte intrauterina,1 posición 1,<5
2,1 solo código de muerte intrauterina,2 posiciones 2 a 3,<5
3,1 solo código de muerte intrauterina,3 posiciones 4 a 10,<5
4,1 solo código de muerte intrauterina,4 posición 11 o después,<5
5,2 solo código de desenlace de mortinato,2 posiciones 2 a 3,23
6,2 solo código de desenlace de mortinato,3 posiciones 4 a 10,21
7,2 solo código de desenlace de mortinato,4 posición 11 o después,<5
8,3 ambas familias,1 posición 1,64
9,3 ambas familias,2 posiciones 2 a 3,11
10,3 ambas familias,3 posiciones 4 a 10,<5


,nivel del evento,código de pérdida temprana,código de aborto,banda de semanas,admisiones
1,solo parcial,No,No,28 o más,8
2,solo parcial,No,No,menos de 22,<5
3,solo parcial,No,No,desconocida,<5
4,estricto,No,No,22 a 27,17
5,estricto,No,No,28 o más,36
6,estricto,No,No,menos de 22,7
7,estricto,No,No,desconocida,54
8,estricto,No,Sí,22 a 27,<5
9,estricto,No,Sí,28 o más,<5
10,estricto,No,Sí,menos de 22,<5


,alcance,admisiones,principal (estricta),estricta + parcial,estricta sin código de pérdida temprana,22 semanas o más,estrictos sin semanas,28 semanas o más
1,1 población D3,"19,440",129,142,119,55,61,37
2,"2 población D3, solo admisiones CIE-10","8,893",71,80,65,55,<5,37
3,"3 población D3, con semanas Z3A","8,852",68,77,63,55,0,37


,nivel del evento,admisiones,terminan antes de 6 h,estancia faltante
1,sin evento,"19,298",<5,0
2,solo parcial,13,0,0
3,estricto,129,0,0


,revisión,resultado,decisión
1,etiqueta principal,129 eventos estrictos,código estricto de muerte fetal a cualquier edad gestacional
2,códigos estrictos más parciales,142 eventos,análisis de sensibilidad
3,estricta sin código de pérdida temprana,119 eventos,análisis de sensibilidad
4,22 semanas o más (solo CIE-10),55 eventos,análisis de sensibilidad
5,28 semanas o más (solo CIE-10),37 eventos,análisis de sensibilidad
6,eventos estrictos sin semanas de gestación,61 eventos,se reportan aparte
7,momento del evento,ninguno: códigos de egreso sin fecha,se mide con indicios en E6 (notebook 03)


## P1 · Puerta de definición — ¿se reproducen los conteos de referencia?

La población, la unidad y el desenlace quedan fijos y los conteos se pueden reproducir. Los conteos de referencia son los de la auditoría de conteos (`count_audit_es.md`). Si alguno difiere, el notebook se detiene.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Se reproducen todos los conteos | P1 pasa: la definición queda fija; se pasa a la auditoría de conteos (notebook 02) | Pasa |
| Un conteo difiere | Se detiene (`assert`): volver a E2 o E3 hasta que la definición sea estable | — |

In [13]:
# Gate G1: reference counts of the count audit (count_audit.md, sections 3.1, 3.6 and 3.7)
t15 = res15["Q-T15d events with the verified table"].iloc[0]
t16 = res16["Q-T16a integrity"].iloc[0]
g1_ok = check([
    ("universe: obstetric admissions", int(p0.n_admissions), 25736),
    ("universe: patients", int(p0.n_patients), 15084),
    ("delivery hospitalizations", int(p0.n_delivery_hospitalizations), 19425),
    ("label table: rows (one per admission)", int(t16.n_rows), 25736),
    ("label table: distinct admissions", int(t16.n_distinct_admissions), 25736),
    ("events, strict or partial (universe)", int(t16.n_events), 154),
    ("strict events (universe)", int(t16.n_strict), 136),
    ("partial-only events (universe)", int(t16.n_partial_only), 18),
    ("population D3", int(t16.n_population_d3), 19440),
    ("population D3 (S2) = population of the label table", int(p0.n_population), int(t16.n_population_d3)),
    ("events in population D3", int(t16.n_events_population_d3), 142),
    ("strict events in population D3 (main label)", int(t16.n_strict_population_d3), 129),
    ("strict events outside population D3", int(t16.n_strict_outside_population), 7),
    ("events with an early-loss code", int(t16.n_events_with_early_loss_code), 11),
    ("verified catalogue: strict events (universe)", int(t15.n_strict), 136),
    ("verified catalogue: strict events in population D3", int(t15.n_strict_population_d3), 129),
], "G1 · Reference counts of the count audit")
decision("G1 · Definition gate", [
    ("population, unit and outcome fixed; counts reproducible", "passes" if g1_ok else "DOES NOT PASS",
     "the definition is fixed: on to the count audit (notebook 02)" if g1_ok else "stop: go back to S2 or S3")])
assert g1_ok, "gate G1 does not pass: a reference count differs"

,conteo,cargado,esperado,resultado
1,universo: admisiones obstétricas,"25,736","25,736",reproducido
2,universo: pacientes,"15,084","15,084",reproducido
3,hospitalizaciones de parto,"19,425","19,425",reproducido
4,tabla de etiqueta: filas (una por admisión),"25,736","25,736",reproducido
5,tabla de etiqueta: admisiones distintas,"25,736","25,736",reproducido
6,"eventos, estrictos o parciales (universo)",154,154,reproducido
7,eventos estrictos (universo),136,136,reproducido
8,eventos solo parciales (universo),18,18,reproducido
9,población D3,"19,440","19,440",reproducido
10,población D3 (E2) = población de la tabla de etiqueta,"19,440","19,440",reproducido


,revisión,resultado,decisión
1,"población, unidad y desenlace fijos; conteos reproducibles",pasa,la definición queda fija: se pasa a la auditoría de conteos (notebook 02)


## Resumen del notebook

Indicadores principales de este notebook y el paso siguiente.

In [14]:
# Summary of notebook 01
notebook_summary([
    ("population of the model (D3)", lambda: f"{int(p0.n_population):,} admissions of {int(p0.n_patients_population):,} patients"),
    ("main label (D5)", "strict fetal-death code at any gestational age"),
    ("strict events in the population", lambda: f"{int(t16.n_strict_population_d3)} ({100 * int(t16.n_strict_population_d3) / int(t16.n_population_d3):.2f}%)"),
    ("events with partial codes (sensitivity)", lambda: int(t16.n_events_population_d3)),
    ("event time in the data", "none: discharge codes without a date"),
    ("gate G1", lambda: "passes" if g1_ok else "DOES NOT PASS"),
    ("next step", "count audit (S4), notebook 02"),
])
write_outputs(OUT_FILE, "Notebook 01 (S1, S2, S3, G1), fetal death MIMIC-IV (M3); aggregate output only")

,valor
indicador,
población del modelo (D3),"19,440 admisiones de 13,735 pacientes"
etiqueta principal (D5),código estricto de muerte fetal a cualquier edad gestacional
eventos estrictos en la población,129 (0.66%)
eventos con códigos parciales (sensibilidad),142
momento del evento en los datos,ninguno: códigos de egreso sin fecha
puerta P1,pasa
paso siguiente,"auditoría de conteos (E4), notebook 02"


salida agregada escrita: data/outputs_01_outcome_definition.txt
